### 回測參數設定
我們可以透過此 Python notebook ，使用不同參數進行回測。以下是各個參數的說明：

| 參數 | 位置 / 範例 | 用途  |
|------|------------|------|
| **backtest_period** | `[30, 60, 90]` | 想測試的所有回測週期，在這裡為存款開放期（天）。若以 60 為例，代表第 1 ~ 60 天允許 deposit，第 61 ~ 90 天只准 withdraw，確保所有 note 都能到期領回。 |
| **init_eths** | `[10, 100]` | 池子啟動時放入的 ETH 數量。對應的 USDC 會依首日 ETH 價格自動計算，維持 50/50 配置。 |  
| **scales** | `[1, 5, 10, 20, 50]` | **用戶下注切割倍率**：<br>‧ 每天投資筆數 × `scale`<br>‧ 每筆金額 ÷ `scale`<br>→ 模擬「把同樣資金拆成很多小單」以觀察滑價與 APR 收斂情況。 |  
| **basis** | `[0.5]` | Dyson Premium 的**基準利率**（0–1）。數值越高，投資人拿到的 premium 越多，池子負擔越大。 |  
| **w_factor** | `[1]` | 池子**目標權重**： `w = k × W_FACTOR`。影響 premium 折扣函式中累積流動性 `q/w` 的斜率，決定 premium 壓抑速度。 |  
| **forward_or_reverse_prob** | `[0.5]` | 使用者每天參與正向或反向雙幣的機率。0.5 表示一半一半。 |  
| **forward_single_side_prob** | `[0.5]` | 在投資正向雙幣時，投資者只買單邊的機率。 |  
| **forward_eth_side_prob** | `[0.5]` | 在正向雙幣且只投單邊時，選擇只投入 ETH 的機率。 |  
| **reverse_single_side_prob** | `[0.5]` | 在投資反向雙幣時，投資者只買單邊選擇權的機率。 |  
| **reverse_eth_side_prob** | `[0.5]` | 在反向雙幣且只投單邊時，選擇買 ETH 方向（CALL）的機率。 |  
| **rebalance_interval** | `[1]` | 每幾天進行一次池子再平衡，單位為天。 |

回測邏輯提醒：
- 進行反向雙幣時，會檢查投入後如果該週期 Q 值必須不為負。若為負，會取消當次反向雙幣並進行下一天的回測。
- 反向雙幣投資者皆在其部位「到期前一天」才檢查價格並決定是否行權。
- 若當天有進行 rebalance，都是在當天的最後一步進行。

In [1]:
import itertools

from backtest_runner import BacktestRunner
from analyzer import Analyzer

# === 參數設定 ===
PARAM_GRID = {
    "backtest_period": [60],
    "init_eths": [100],
    "scale": [1],
    "basis": [0.7],
    "w_factor": [1],
    "forward_or_reverse_prob": [0.5],
    "forward_single_side_prob": [0.5],
    "forward_eth_side_prob": [0.5],
    "reverse_single_side_prob": [0.5],
    "reverse_eth_side_prob": [0.5],
    "rebalance_interval": [1], 
}

# 參數縮寫對照表
param_alias = {
    "backtest_period": "T",
    "init_eths": "E",
    "scale": "S",
    "basis": "B",
    "w_factor": "W",
    "forward_or_reverse_prob": "FR",
    "forward_single_side_prob": "FS",
    "forward_eth_side_prob": "FE",
    "reverse_single_side_prob": "RS",
    "reverse_eth_side_prob": "RE",
    "rebalance_interval": "RI",
}

# 預設值字典（用於判斷哪些參數是預設，不顯示在 tag 中）
default_config = {
    "backtest_period": 60,
    "init_eths": 100,
    "scale": 1,
    "basis": 0.7,
    "w_factor": 1,
    "forward_or_reverse_prob": 0.5,
    "forward_single_side_prob": 0.5,
    "forward_eth_side_prob": 0.5,
    "reverse_single_side_prob": 0.5,
    "reverse_eth_side_prob": 0.5,
    "rebalance_interval": 1,
}

# 所有組合
param_names = list(PARAM_GRID.keys())
param_values = list(PARAM_GRID.values())
all_combinations = list(itertools.product(*param_values))


# === 標籤產生函數 ===
def build_tag(param_dict, info, alias_map, default_dict=None):
    parts = []
    for k, v in param_dict.items():
        if default_dict and default_dict.get(k) == v:
            continue  # 不顯示與預設相同的參數
        alias = alias_map.get(k, k)
        parts.append(f"{alias}{v}")
    short_tag = "-".join(parts)
    date_range = f"{info['start_date']}~{info['total_end_date']}"
    return f"{short_tag} ({date_range})"


# === 主執行流程 ===
results = {}

for param_tuple in all_combinations:
    param_dict = dict(zip(param_names, param_tuple))

    runner = BacktestRunner(
        init_eth=param_dict["init_eths"],
        main_days=param_dict["backtest_period"],
        basis=param_dict["basis"],
        scale=param_dict["scale"],
        forward_or_reverse_prob=param_dict["forward_or_reverse_prob"],
        forward_single_side_prob=param_dict["forward_single_side_prob"],
        forward_eth_side_prob=param_dict["forward_eth_side_prob"],
        reverse_single_side_prob=param_dict["reverse_single_side_prob"],
        reverse_eth_side_prob=param_dict["reverse_eth_side_prob"],
        rebalance_interval=param_dict["rebalance_interval"],
    )

    info = runner.get_info()

    # 產生簡短 tag（用於檔名與 title）
    tag = build_tag(param_dict, info, param_alias, default_dict=default_config)

    # 執行回測
    dep_df, wd_df, reverse_dep_df, snap_df = runner.run()
    results[tuple(param_tuple)] = (dep_df, wd_df, reverse_dep_df, snap_df, info)

    # 顯示資訊
    print(f"========= {tag} =========")
    print(f"deposits: {len(dep_df)}  withdraws: {len(wd_df)}")
    print(f"reverse deposits: {len(reverse_dep_df)}")

    # 繪圖分析
    Analyzer(dep_df, wd_df, reverse_dep_df, snap_df, tag).all_plots()



/Users/lichengwang/DysonV2-Python/backtest_module/backtest_runner.py:284: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.deposits = pd.concat(
/Users/lichengwang/DysonV2-Python/backtest_module/backtest_runner.py:398: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.daily = pd.concat([self.daily, new_daily], ignore_index=True)
/Users/lichengwang/DysonV2-Python/backtest_module/backtest_runner.py:355: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is depre

=========  (2025-04-10~2025-07-08) =========
deposits: 151  withdraws: 151
reverse deposits: 41
